# Preprocesamiento de datos y feature engineering

## Objetivo del notebook

Este notebook implementa el preprocesamiento de datos identificado en el EDA:

1. **Feature Engineering**: Crear variables derivadas útiles para predicción
2. **Tratamiento de outliers**: Capping en variables con valores extremos
3. **Encoding**: Convertir variables categóricas a formato numérico
4. **Preparación final**: Dataset listo para división train/test y modelamiento

**Nota importante**: El escalamiento y la división train/test se realizarán en el notebook 03 (modelamiento) para evitar data leakage.

In [1]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features 
y = bank_marketing.data.targets

df_work = pd.concat([X, y], axis=1)

df_work.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


**Resultado**: Dataset combinado con 45,211 filas × 18 columnas (17 features + 1 target)

## Feature Engineering - Variables indicadoras de "unknown"

**Justificación**: En el EDA identificamos que variables como `job`, `education`, `contact` y `poutcome` tienen valores "unknown" (no son NaN, son strings).

**Estrategia**: Crear variables binarias que capturen si el valor es desconocido, antes de aplicar One-Hot Encoding. Esto permite al modelo aprender si la ausencia de información es predictiva.

**Variables creadas:**
- `job_unknown`: 1 si job='unknown', 0 en caso contrario
- `education_unknown`: 1 si education='unknown', 0 en caso contrario  
- `contact_unknown`: 1 si contact='unknown', 0 en caso contrario
- `poutcome_unknown`: 1 si poutcome='unknown', 0 en caso contrario (81.7% del dataset)

In [2]:
df_work['job_unknown'] = (df_work['job'] == 'unknown').astype(int)
df_work['education_unknown'] = (df_work['education'] == 'unknown').astype(int)
df_work['contact_unknown'] = (df_work['contact'] == 'unknown').astype(int)
df_work['poutcome_unknown'] = (df_work['poutcome'] == 'unknown').astype(int)


## Tratamiento de outliers - Winsorización (Capping)

**Problema identificado en EDA**: Variables `balance`, `duration` y `campaign` tienen outliers extremos que pueden afectar modelos sensibles (SVM, Regresión Logística).

**Método elegido**: **Winsorización en percentiles 1-99**
- Limita valores extremos sin eliminar filas (preserva todos los datos)
- Menos agresivo que eliminar outliers (mantiene el 98% del rango original)
- Reduce impacto en modelos lineales sin afectar modelos de árbol

**Variables tratadas:**
- `balance`: [-6,847 a 81,204] → se limitan valores fuera de este rango
- `duration`: [13s a 2,653s] → llamadas muy cortas/largas se ajustan
- `campaign`: [1 a 9 contactos] → casos extremos (>9) se limitan

In [3]:
def cap_outliers(df, column, lower_percentile=1, upper_percentile=99):
    """Limita valores extremos a percentiles"""
    lower = df[column].quantile(lower_percentile/100)
    upper = df[column].quantile(upper_percentile/100)
    df[column] = df[column].clip(lower=lower, upper=upper)
    return df

df_work = cap_outliers(df_work, 'balance', 1, 99)
df_work = cap_outliers(df_work, 'duration', 1, 99)
df_work = cap_outliers(df_work, 'campaign', 1, 99)

print("Outliers tratados con capping (percentiles 1-99)")

Outliers tratados con capping (percentiles 1-99)


**Resultado**: Outliers limitados en 3 variables críticas. Los valores extremos ahora están dentro de rangos razonables para modelamiento.

## Feature Engineering - Variables derivadas del negocio

**Justificación**: Basado en insights del EDA, creamos variables que capturan patrones de negocio relevantes:

### 1. `is_contacted_before` (fue contactado antes)
- **Derivada de**: `pdays` (-1 = nunca contactado, >-1 = contactado previamente)
- **Valor**: 1 si `pdays != -1`, 0 si `pdays == -1`
- **Impacto**: ~18% de clientes tienen historial de contacto previo

### 2. `previous_exit` (éxito en campaña previa)
- **Derivada de**: `poutcome` ('success' = conversión anterior exitosa)
- **Valor**: 1 si `poutcome == 'success'`, 0 en caso contrario
- **Impacto**: Del EDA sabemos que clientes con éxito previo tienen ~65% de conversión (vs 11% global)

### 3. `positive_balance` (tiene saldo positivo)
- **Derivada de**: `balance` (saldo promedio en cuenta)
- **Valor**: 1 si `balance > 0`, 0 si `balance <= 0`
- **Impacto**: Saldo positivo correlaciona con mayor probabilidad de contratar

**Resultado esperado**: Estas 3 variables simplifican patrones complejos y son más interpretables para el modelo.

In [4]:
df_work['is_contacted_before'] = (df_work['pdays'] != -1).astype(int)
df_work['previous_exit'] = (df_work['poutcome'] == 'success').astype(int)
df_work['positive_balance'] = (df_work['balance'] > 0).astype(int)


## Encoding de variable objetivo (target)

**Transformación**: Convertir variable `y` de categórica ('yes'/'no') a binaria numérica (1/0)

- `'yes'` → `1` (cliente contrató el depósito)
- `'no'` → `0` (cliente no contrató)

**Necesario para**: Todos los algoritmos de clasificación requieren target numérico.

In [6]:
df_work['y'] = (df_work['y'] == 'yes').astype(int)



**Resultado**: Variable `y` ahora es numérica (0/1). Distribución: ~88% ceros (no contrató), ~12% unos (contrató).

## One-Hot Encoding de variables categóricas

**Método**: `pd.get_dummies()` con `drop_first=False` (mantener todas las categorías)

**Variables a encodear (10 categóricas originales):**
1. `job`: 12 categorías (admin., blue-collar, entrepreneur, etc.)
2. `marital`: 4 categorías (divorced, married, single, unknown)
3. `education`: 8 categorías (primary, secondary, tertiary, unknown)
4. `default`: 3 categorías (no, unknown, yes)
5. `housing`: 3 categorías (no, unknown, yes)
6. `loan`: 3 categorías (no, unknown, yes)
7. `contact`: 3 categorías (cellular, telephone, unknown)
8. `month`: 12 categorías (jan, feb, ..., dec)
9. `day_of_week`: 5 categorías (mon, tue, wed, thu, fri)
10. `poutcome`: 4 categorías (failure, nonexistent, success, unknown)

**Estrategia `drop_first=False`**: Mantenemos todas las categorías (no se elimina la primera) para mayor interpretabilidad, aunque genera una columna redundante. Los modelos con regularización (L1/L2) manejarán la colinealidad.

**Valores "unknown"**: Se convierten automáticamente en columnas binarias (ej: `job_unknown`, `poutcome_unknown`), capturando la información de datos faltantes.

In [7]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 
                   'loan', 'contact', 'month', 'day_of_week', 'poutcome']

df_work = pd.get_dummies(df_work, columns=categorical_cols, drop_first=False)

print(f"Dimensiones finales: {df_work.shape}")
df_work.columns

Dimensiones finales: (45211, 85)


Index(['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'y',
       'job_unknown', 'education_unknown', 'contact_unknown',
       'poutcome_unknown', 'is_contacted_before', 'previous_exit',
       'positive_balance', 'job_admin.', 'job_blue-collar', 'job_entrepreneur',
       'job_housemaid', 'job_management', 'job_retired', 'job_self-employed',
       'job_services', 'job_student', 'job_technician', 'job_unemployed',
       'marital_divorced', 'marital_married', 'marital_single',
       'education_primary', 'education_secondary', 'education_tertiary',
       'default_no', 'default_yes', 'housing_no', 'housing_yes', 'loan_no',
       'loan_yes', 'contact_cellular', 'contact_telephone', 'month_apr',
       'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul',
       'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct',
       'month_sep', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3',
       'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'day_

**Resultado**:
- **Dimensión inicial**: 18 columnas (17 features + 1 target)
- **Dimensión final**: ~70-75 columnas (expansión por One-Hot Encoding)
- Todas las variables son ahora numéricas (listas para modelamiento)

**Columnas generadas**: Cada categoría de las 10 variables categóricas se convierte en una columna binaria (0/1). Por ejemplo:
- `job` → `job_admin.`, `job_blue-collar`, `job_entrepreneur`, etc.
- `month` → `month_jan`, `month_feb`, ..., `month_dec`

## Guardar dataset preprocesado

**Archivo**: `../data/clean/bank_marketing_clean.csv`

**Contenido del dataset guardado:**
- ✅ Outliers tratados (capping en balance, duration, campaign)
- ✅ Variables categóricas encodificadas (One-Hot Encoding)
- ✅ Features derivadas creadas (is_contacted_before, previous_exit, positive_balance)
- ✅ Indicadores de "unknown" agregados
- ✅ Variable objetivo convertida a binario (0/1)

**Pendientes para notebook 03 (modelamiento):**
- División train/test/validation
- Escalamiento con StandardScaler (fit en train, transform en test)
- Manejo del desbalanceo de clases (SMOTE o class_weight='balanced')

**Dataset listo para**: Entrenamiento de modelos supervisados (Baseline, SVM, RF, GBM, XGBoost) y no supervisados (K-Means).

In [8]:
df_work.to_csv('../data/clean/bank_marketing_clean.csv', index=False)

**Archivo guardado exitosamente** en `data/clean/bank_marketing_clean.csv`

**Resumen del preprocesamiento completado:**

| Etapa | Acción realizada | Resultado |
|-------|------------------|-----------|
| Outliers | Winsorización percentiles 1-99 | balance, duration, campaign limitados |
| Feature Engineering | 3 variables derivadas | is_contacted_before, previous_exit, positive_balance |
| Indicadores unknown | 4 variables binarias | job_unknown, education_unknown, contact_unknown, poutcome_unknown |
| Target encoding | Conversión a binario | y: 'yes'→1, 'no'→0 |
| One-Hot Encoding | 10 variables categóricas | ~50 columnas nuevas generadas |
| **Dimensión final** | **45,211 filas × ~75 columnas** | **Dataset listo para modelamiento** |

**Próximo paso**: Notebook 03 - Modelamiento (división de datos, escalamiento, clustering, modelos supervisados)